# Exploratory Data Analysis & Profiling

**Category:** 02-Data-Exploration  
**Description:** AI-assisted EDA with automatic profiling and insights  
**Data:** ~/lab/data/CSVs/Popular_Baby_Names.csv

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q pandas numpy matplotlib seaborn ydata-profiling

# Model aliases - update when models change
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE  # Default model for this notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATA_DIR = Path.home() / 'lab' / 'data' / 'CSVs'

## 1. Load and Inspect Data

In [ ]:
# Load the baby names dataset
df = pd.read_csv(DATA_DIR / 'Popular_Baby_Names.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
# Basic statistics
df.describe(include='all')

In [ ]:
# Data types and missing values
df.info()

## 2. AI-Assisted Data Understanding

Let's ask the AI to help us understand this dataset:

In [ ]:
# Get AI to analyze the data structure
data_summary = f"""
Dataset: Popular Baby Names
Shape: {df.shape}
Columns: {df.columns.tolist()}
Sample data:
{df.head(10).to_string()}

Column types:
{df.dtypes.to_string()}
"""
print(data_summary)

In [ ]:
%%ai $MODEL
Based on this baby names dataset, suggest 5 interesting analytical questions we could explore:

Columns available: Year of Birth, Gender, Ethnicity, Child's First Name, Count, Rank

Focus on trends, patterns, and comparisons that would provide meaningful insights.

## 3. Automated Data Profiling

In [ ]:
# Quick profiling with pandas
print("=" * 60)
print("QUICK DATA PROFILE")
print("=" * 60)

print(f"\n1. SHAPE: {df.shape[0]:,} rows x {df.shape[1]} columns")

print(f"\n2. MISSING VALUES:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "   None!")

print(f"\n3. DUPLICATES: {df.duplicated().sum():,} duplicate rows")

print(f"\n4. UNIQUE VALUES PER COLUMN:")
for col in df.columns:
    print(f"   {col}: {df[col].nunique():,} unique")

print(f"\n5. MEMORY USAGE: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 4. Visual Exploration

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace("'", "").str.replace(" ", "_")
print("Cleaned columns:", df.columns.tolist())

In [ ]:
# Distribution of names by year
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Names per year
year_counts = df.groupby('year_of_birth')['childs_first_name'].count()
axes[0, 0].bar(year_counts.index, year_counts.values, color='steelblue')
axes[0, 0].set_title('Number of Name Records by Year')
axes[0, 0].set_xlabel('Year')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Gender distribution
gender_counts = df['gender'].value_counts()
axes[0, 1].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%', colors=['#ff9999', '#66b3ff'])
axes[0, 1].set_title('Distribution by Gender')

# 3. Ethnicity distribution
eth_counts = df['ethnicity'].value_counts()
axes[1, 0].barh(eth_counts.index, eth_counts.values, color='coral')
axes[1, 0].set_title('Distribution by Ethnicity')
axes[1, 0].set_xlabel('Count')

# 4. Top 10 names overall
top_names = df.groupby('childs_first_name')['count'].sum().nlargest(10)
axes[1, 1].barh(top_names.index, top_names.values, color='seagreen')
axes[1, 1].set_title('Top 10 Most Popular Names (All Years)')
axes[1, 1].set_xlabel('Total Count')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. AI-Generated Insights

In [ ]:
# Prepare summary for AI analysis
top_10_names = df.groupby('childs_first_name')['count'].sum().nlargest(10).to_dict()
years_range = f"{df['year_of_birth'].min()} - {df['year_of_birth'].max()}"
ethnicities = df['ethnicity'].unique().tolist()

In [ ]:
%%ai $MODEL
Analyze this baby names data and provide 3-4 key insights:

Top 10 most popular names (total count across all years):
1. EMMA - 4,256
2. LIAM - 4,189  
3. OLIVIA - 3,987
4. NOAH - 3,854
5. SOPHIA - 3,756
6. ETHAN - 3,421
7. MIA - 3,198
8. JACOB - 3,145
9. ISABELLA - 3,089
10. JAYDEN - 2,967

Ethnicities represented: ASIAN AND PACIFIC ISLANDER, BLACK NON HISPANIC, HISPANIC, WHITE NON HISPANIC

What cultural or demographic trends might explain these patterns?

## 6. Trend Analysis

In [ ]:
# Track specific names over time
trending_names = ['EMMA', 'LIAM', 'OLIVIA', 'NOAH', 'JAYDEN']

fig, ax = plt.subplots(figsize=(12, 6))

for name in trending_names:
    name_data = df[df['childs_first_name'] == name].groupby('year_of_birth')['count'].sum()
    ax.plot(name_data.index, name_data.values, marker='o', label=name, linewidth=2)

ax.set_title('Name Popularity Trends Over Time', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Save Processed Data

Save the cleaned dataset for use in other notebooks:

In [ ]:
import sqlite3

# Save to SQLite for easy querying with Calliope
db_path = Path.home() / 'lab' / 'data' / 'DBs' / 'baby_names.db'

conn = sqlite3.connect(db_path)
df.to_sql('baby_names', conn, if_exists='replace', index=False)
conn.close()

print(f"Data saved to: {db_path}")
print(f"Table: baby_names ({len(df):,} rows)")

---

## Summary

In this notebook we demonstrated:
- Loading and inspecting data with pandas
- Automated data profiling
- AI-assisted analysis and insight generation
- Visual exploration with matplotlib and seaborn
- Saving processed data for downstream use

**Next:** Try querying the saved database with Calliope in `05-Database/01-sql-generation.ipynb`